# Submission из завершённого checkpoint

Поддерживаются LR (.joblib), LightGBM (.txt), CatBoost (.cbm). Для деревьев требуется также готовый test-кэш той же поставки; сигналы и статистики test не используются для обучения. CSV следует точному порядку sample_submission.csv. Выберите EXPERIMENT_ID и выполните ячейки.


In [ ]:
import sys
from pathlib import Path

# На Kaggle задайте абсолютный путь к каталогу кода проекта.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if (
    not (PROJECT_ROOT / "pyproject.toml").is_file()
    or not (PROJECT_ROOT / "src" / "__init__.py").is_file()
):
    raise FileNotFoundError("Укажите PROJECT_ROOT: каталог с pyproject.toml и src/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.notebook import run_command

EXPERIMENT_ID = "015_tree"  # Или завершённый 009_tree ... 014_tree.
folder = PROJECT_ROOT / "artefacts" / EXPERIMENT_ID
checkpoints = [
    folder / f"tuned.{ext}"
    for ext in ("joblib", "txt", "cbm")
    if (folder / f"tuned.{ext}").is_file()
]
if not checkpoints:
    checkpoints = [
        folder / f"fixed.{ext}"
        for ext in ("joblib", "txt", "cbm")
        if (folder / f"fixed.{ext}").is_file()
    ]
if len(checkpoints) != 1:
    raise FileNotFoundError(f"Нужен один завершённый tuned checkpoint: {folder}")
CHECKPOINT = checkpoints[0]
DATA_ROOT = PROJECT_ROOT / "data/raw"
if not (DATA_ROOT / "labels.csv").is_file():
    DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT = PROJECT_ROOT / "submissions" / f"{EXPERIMENT_ID}.csv"
run_command(
    [
        "uv",
        "run",
        "--locked",
        "python",
        "-m",
        "src.inference",
        str(CHECKPOINT),
        "--data-root",
        str(DATA_ROOT),
        "--output",
        str(OUTPUT),
    ],
    cwd=PROJECT_ROOT,
)

Сборщик проверяет точный набор ID и порядок sample_submission.csv. Рядом сохраняется .meta.yaml с ID эксперимента, C, SHA-256 checkpoint и версией признаков. Тестовые сигналы используются только для фиксированного извлечения признаков и контекста внутри непрерывной записи.